# 00 総合ガイド — llmlab の使い方ひととおり

OpenAI 互換エンドポイント（ローカルLLM）を JupyterLab で使うツールキット。
このノートブックは **接続設定 → 各機能** を一通り体験する入口です。各機能の詳細は
番号付きノートブック（01〜07）を参照してください。

| 機能 | 何ができる | 詳細 |
|------|-----------|------|
| ① コード補完 | セルのコードを補完 | 01 |
| ② チャット | 対話・コード生成 | 01 |
| ③ build_rag | 文書をざっと検索 | 02 |
| ④ PagedRAG | ページ出典つき検索 | 03 |
| ⑤ BookRAG | 構造・用語関係まで深掘り | 04 |
| ⑥ MultiPaperRAG | 複数論文の横断比較 | 05 |
| ⑦ TableQA | 表の集計・計算（text-to-pandas） | 06 |
| ⑧ DocQA | 1文書内で 散文=RAG / 表=TableQA を自動振り分け | 07 |

> ①② はチャットAPIだけで動く。③〜⑥（RAG）は埋め込みAPI（/v1/embeddings）が必要。
> ⑦ TableQA はチャットAPIのみ。⑧ DocQA は本文RAGに埋め込みが必要。

## 0. インストール（初回のみ）

```bash
cd jupyter-local-llm
pip install -e .                 # 基本（①〜⑤の中心機能）
# 任意機能:
# pip install -e ".[tables,office,figures,local-embeddings]"
# pip install pandas             # Comparison.to_df()
```

## 1. 接続設定（最初に1回 / セッションごと）

接続情報は**メモリ上のみ**でファイルに保存しません。フォーム or コードで入力します。

In [ ]:
import llmlab
llmlab.settings_form()    # Base URL / API Key / Model / Embed / プロキシ を入力

In [ ]:
# コードで直接設定する場合（フォームが出ない環境でも確実）
# llmlab.configure(
#     base_url="http://localhost:8000/v1",
#     api_key="...",
#     model="your-chat-model",
#     embed_model="your-embedding-model",     # RAG 用
#     # embed_base_url="http://embed-host/v1", # 埋め込みが別エンドポイントなら
#     # embed_provider="local",                # サーバに /v1/embeddings が無いとき
# )

llmlab.doctor()    # 環境診断（ウィジェットが出ない/依存不足の切り分け）

## 2. ① コード補完

関数 `code_complete` / マジック `%%complete` / UI `completion_panel()` の3通り。
入力中のゴーストテキスト補完は同梱の JupyterLab 拡張（`labextension/`）で有効化。

In [ ]:
from llmlab import code_complete
print(code_complete("def fib(n):\n"))

In [ ]:
%load_ext llmlab.complete

In [ ]:
%%complete
def quicksort(arr):

## 3. ② チャット

単発 `complete()` / 履歴つきマジック `%%llm` / UI `chat_panel()`。

In [ ]:
from llmlab import complete
print(complete("pandas で CSV を読み込むコードを書いて"))

In [ ]:
%load_ext llmlab.chat

In [ ]:
%%llm
この関数をベクトル化して高速化して

In [ ]:
# ノートブック内チャットUI
llmlab.chat_panel()

## 4. ③ 汎用 RAG（`build_rag`）

`docs/` 配下やファイルを取り込み、ざっと検索。埋め込み API が必要。

In [ ]:
# フォルダでもファイル単体でも可
engine = llmlab.build_rag("../docs")
print(engine.query("この資料の要点は？"))

## 5. ④ PagedRAG（ページ出典つき）

文書単位で管理し、回答に書名・ページの出典が付く。

In [ ]:
rag = llmlab.PagedRAG()        # DocRAG は別名
rag.add_book("../docs/営業マニュアル.pdf", title="営業マニュアル")
print(rag.ask("返品の手順は？"))

## 6. ⑤ BookRAG（構造・用語関係まで深掘り）

目次ツリー＋知識グラフ（BookIndex）を作り、質問の種類に応じて検索戦略を変える。
取り込み(`add_book`)は本文ノードごとに LLM 抽出が走るため**重い**（長尺文書で数分〜）。

In [ ]:
book = llmlab.BookRAG()
book.add_book("../docs/handbook.pdf", title="Handbook")
print(book.info())            # ノード/エンティティ/関係数（0 なら抽出に失敗）
print(book.ask("How does X differ from Y?"))

## 7. ⑥ MultiPaperRAG（複数論文の横断比較）

「広く探す → 論文ごとに深掘り → 突き合わせ」。PDF/Word/Excel 対応。
**`pics=False`（既定）なら VLM 不使用**（埋め込み検索のみ）。図比較が要るときだけ `pics=True`。

In [ ]:
mp = llmlab.MultiPaperRAG(
    deep_engine="paged",        # "paged"(速い) / "book"(構造的に深掘り)
    locate_strategy="search",   # "search" / "summary" / "all"
    pics=False,                 # True で図理解（要 pymupdf + 画像対応モデル）
)
mp.add_papers("../papers")
cmp = mp.compare("提案手法の精度を論文間で比較して")
print(cmp)

In [ ]:
cmp.to_df()                               # 論文ごとの結果を DataFrame に（要 pandas）
# print(mp.compare_table("ImageNet accuracy"))   # 表の数値を横断比較

## 8. ⑦ TableQA（表の集計・計算 / text-to-pandas）

Excel/CSV への「合計・平均・件数・条件抽出」を pandas コード生成で解く。RAG が苦手な計算担当。

In [ ]:
tq = llmlab.TableQA("../docs/売上.xlsx")   # CSV / DataFrame / {name: DataFrame} も可
print(tq.ask("東京支店の4月の売上合計は？"))     # 回答 + 生成コード + 実行結果

## 9. ⑧ DocQA（1文書を 散文=RAG / 表=TableQA に自動振り分け）

同じファイル内で、本文は RAG、表は TableQA に自動ルーティング（`route="rag"/"table"` で強制も可）。

In [ ]:
doc = llmlab.DocQA("../docs/report.pdf")   # PDF / Excel / Word / CSV
print(doc.ask("この資料の要点を3つ"))          # → 本文RAG
print(doc.ask("売上の合計と平均は？"))           # → 表TableQA
doc.table_names()                             # 抽出された表

---
## 困ったとき

- フォーム/パネルが `VBox(...)` のテキストになる → `llmlab.doctor()`、または `settings_form(text=True)`
- RAG で `404 /v1/embeddings` → サーバに埋め込みが無い。`configure(embed_base_url=...)` か
  `configure(embed_provider="local")`（要 `pip install -e ".[local-embeddings]"`）
- BookRAG/MultiPaperRAG が遅い → 設計上 LLM 呼び出しが多い。`build_rag`/`PagedRAG` で十分なら速い
- `info()` の entities が 0 → モデルが JSON を返していない可能性（`complete('Return JSON: {\"ok\":true}')` で確認）